# Step 8: Model 5 - PyTorch with Lightning Flash (TabularRegressor).

Lightning Flash is archived (no longer maintained) and only installs on older
Python/PyTorch. Run this script from the separate Python 3.10 environment
described in README / requirements-flash.txt.

In [1]:
# Run from the project root so data/, figures/ and results/ paths work
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%matplotlib inline

In [2]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import flash
from flash.tabular import TabularRegressionData, TabularRegressor
from pytorch_lightning.callbacks import EarlyStopping

from config import RANDOM_STATE, TARGET
from utils import load_data, evaluate, save_results, validation_split

torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

X_train, X_test, y_train, y_test, X_latest = load_data()
X_tr, X_val, y_tr, y_val = validation_split(X_train, y_train)
y_col = f"{TARGET}_fwd_ret"
features = list(X_train.columns)
SCALE = 100          # train on returns in PERCENT (raw ~0.03 values are too small to learn well)


def frame(X, y=None):
    df = X.reset_index(drop=True).copy()
    if y is not None:
        df[y_col] = y.values * SCALE
    return df

Train: 2010-03-31 -> 2023-06-23 (667 rows)
Test : 2023-06-30 -> 2026-10-21 (167 rows)


In [3]:
# Flash normalises the numeric columns itself, using the training data statistics
datamodule = TabularRegressionData.from_data_frame(
    numerical_fields=features, target_field=y_col,
    train_data_frame=frame(X_tr, y_tr), val_data_frame=frame(X_val, y_val),
    batch_size=32,
)

In [4]:
# "category_embedding" = a plain feed-forward network from pytorch-tabular
model = TabularRegressor.from_data(datamodule, backbone="category_embedding",
                                   learning_rate=1e-3)
trainer = flash.Trainer(max_epochs=50, enable_progress_bar=False, logger=False,
                        enable_checkpointing=False,
                        callbacks=[EarlyStopping(monitor="valid_loss", patience=10)])

start = time.time()
trainer.fit(model, datamodule=datamodule)
train_seconds = time.time() - start


def predict(X):
    dm = TabularRegressionData.from_data_frame(
        predict_data_frame=frame(X), parameters=datamodule.parameters, batch_size=256)
    batches = trainer.predict(model, datamodule=dm, output="preds")
    return np.concatenate([np.ravel(np.asarray(b, dtype=float)) for b in batches]) / SCALE


y_pred = predict(X_test)
metrics = evaluate(y_test, y_pred, y_train)
expected = predict(X_latest)
save_results("lightning_flash", metrics, y_test, y_pred, expected, train_seconds)

Using 'category_embedding' provided by manujosephv/PyTorch Tabular (https://github.com/manujosephv/pytorch_tabular).


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


IPU available: False, using: 0 IPUs


HPU available: False, using: 0 HPUs



  | Name          | Type                  | Params
--------------------------------------------------------
0 | train_metrics | ModuleDict            | 0     
1 | val_metrics   | ModuleDict            | 0     
2 | test_metrics  | ModuleDict            | 0     
3 | adapter       | PytorchTabularAdapter | 11.9 K
--------------------------------------------------------
11.9 K    Trainable params
0         Non-trainable params
11.9 K    Total params
0.048     Total estimated model params size (MB)


  model                            lightning_flash
  RMSE                             0.0373
  MAE                              0.0262
  R2                               0.0056
  Directional_Accuracy             0.6168
  Baseline_RMSE                    0.0374
  Baseline_MAE                     0.0256
  Baseline_Directional_Accuracy    0.5629
  train_seconds                    1.1400
  expected_weekly_log_return       0.0048
  expected_weekly_return_pct       0.4799
  expected_annualised_return_pct   28.2682


{'model': 'lightning_flash',
 'RMSE': 0.037250390261870846,
 'MAE': 0.026238783920520533,
 'R2': 0.005613678791776988,
 'Directional_Accuracy': 0.6167664670658682,
 'Baseline_RMSE': 0.03737689852806529,
 'Baseline_MAE': 0.02558297501644671,
 'Baseline_Directional_Accuracy': 0.562874251497006,
 'train_seconds': 1.14,
 'expected_weekly_log_return': 0.0047875592112541195,
 'expected_weekly_return_pct': 0.47990378838189685,
 'expected_annualised_return_pct': 28.268184692391984}